<a href="https://colab.research.google.com/github/Rachani02/Statistical-Learning-e22282/blob/main/Assignment%207d%3A%20Structural%20Health%20Monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

1. Prior Belief Boundaries

For $\Theta \sim \text{Beta}(\alpha, \beta)$ with $\alpha = 8$ and $\beta = 1.5$:$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.8421$$

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_grid = np.linspace(0.01, 1.0, 500)
prior_pdf = stats.beta.pdf(theta_grid, 8, 1.5)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=prior_pdf, mode='lines', name='Beta(8, 1.5) Prior', line=dict(color='teal', width=3)))
fig.update_layout(
    title="Initial Structural Health Prior Density: Beta(8, 1.5)",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)
fig.show()

Engineering Rationale

The distribution places nearly all probability mass on $\theta > 0.60$, with a sharp drop-off near zero and a peak near $\theta \approx 0.93$. This reflects realistic domain knowledge: new or operational structural components are overwhelmingly likely to be healthy ($\theta \approx 1.0$), while severe degradation ($\theta < 0.5$) is extremely rare prior to damage events.

---


2. Structural Likelihood Formulation

The measurement model is:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

Taking the natural logarithm:

$$\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k \implies \ln(y_k) \sim \mathscr{N}\left(\ln(\theta \cdot K_{\text{nominal}}), \sigma^2\right)$$

Applying the change of variables transformation $f_{Y_k}(y_k) = f_{\ln(y_k)}(\ln(y_k)) \cdot \left\vert{}\frac{d \ln(y_k)}{dy_k}\right\vert{} = f_{\ln(y_k)}(\ln(y_k)) \cdot \frac{1}{y_k}$, the log-normal single-measurement likelihood contribution is:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$

For the running observation vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$, conditional independence yields the joint likelihood:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\ln(y_i) - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$

---

3. Mathematical Formulation of the Non-Conjugate Grid Update

Why Closed-Form Solutions Fail

The prior distribution belongs to the Beta family (algebraic polynomial form $\theta^{\alpha-1}(1-\theta)^{\beta-1}$), whereas the likelihood function is log-normal (transcendental function of $\ln(\theta)$ inside a Gaussian exponential kernel).Multiplying these kernel forms results in an intractable posterior density:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{\alpha-1}(1-\theta)^{\beta-1} \cdot \exp\left( -\frac{\left(\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$

Because the log-normal likelihood is non-conjugate to the Beta prior, the marginal likelihood constant $Z_k = \int_0^1 L(y_k \mid \theta) f(\theta) d\theta$ cannot be integrated analytically.Recursive Relationship

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

---

4. Running Point Estimates

The point estimators are defined on the bounded physical domain $\theta \in (0, 1]$ by:

1. Running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)

$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

2. Running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \operatorname{arg\,max}_{\theta \in (0, 1]} \left\{ f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \right\}$$

---


5. Numerical Implementation via Bounded Grid Discretization

Since non-linear structural response functions $g(\theta)$ generally lack closed-form analytical conjugate solutions, the system maintains the posterior distribution on a fine discrete grid across the bounded physical domain $[\theta_{\text{min}}, \theta_{\text{max}}]$.Algorithmic Procedure

1. Grid SetupDefine $M$ equally spaced grid points across the bounded domain $[\theta_{\text{min}}, \theta_{\text{max}}]$. The step size $\Delta\theta$ and individual grid points $\theta_m$ are calculated as:

$$\Delta\theta = \frac{\theta_{\text{max}} - \theta_{\text{min}}}{M - 1}$$

$$\theta_m = \theta_{\text{min}} + (m - 1)\Delta\theta, \quad \text{for } m = 1, 2, \dots, M$$

2. Prior InitializationEvaluate the initial prior probability density values at each point across the grid array $P_0 = [P_0(\theta_1), P_0(\theta_2), \dots, P_0(\theta_M)]$. Normalize this discrete array using numerical integration (such as the composite trapezoidal rule):

$$Z_0 = \text{trapezoid}(P_0, \theta)$$

$$P_0 \leftarrow \frac{P_0}{Z_0}$$

3. Sequential Updating & Normalization (at step $k$)Compute Unnormalized Posterior Array:

Multiply the previous step's posterior by the likelihood of the new measurement $y_k$ at each grid point:$$\tilde{P}_k(\theta_m) = P_{k-1}(\theta_m) \times L(y_k \mid \theta_m)$$

Compute Normalizing Factor: Integrate the unnormalized posterior over the grid using the trapezoidal rule:

$$Z_k = \sum_{m=1}^{M-1} \left( \frac{\tilde{P}_k(\theta_m) + \tilde{P}_k(\theta_{m+1})}{2} \right) \Delta\theta$$

Normalize: Scale the values so the total probability across the domain equals 1:

$$P_k(\theta_m) = \frac{\tilde{P}_k(\theta_m)}{Z_k}$$

4. Point Estimation EvaluationBayes Estimate (Posterior Mean): Calculated by taking the expectation of $\theta$ over the normalized posterior grid:

$$\hat{\theta}_{\text{Bayes}}^{(k)} \approx \text{trapezoid}(\theta \odot P_k, \theta)$$

(where $\odot$ represents element-wise multiplication of the grid coordinate array $\theta$ and the posterior array $P_k$)Maximum A Posteriori (MAP) Estimate: Identified as the grid point $\theta_{m^*}$ where the normalized posterior value reaches its maximum:

$$\hat{\theta}_{\text{MAP}}^{(k)} = \theta_{m^*}, \quad \text{where } m^* = \operatorname*{argmax}_m P_k(\theta_m)$$

---

6. Performance Tracking & Degradation Simulation Script

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set seed for reproducibility
np.random.seed(2026)

# Simulation Parameters
n_steps = 15
K_nominal = 50.0  # kN/mm
sigma = 0.15
theta_true = 0.68

# Grid Setup
M = 1000
theta_grid = np.linspace(0.001, 1.0, M)
d_theta = theta_grid[1] - theta_grid[0]

# 1. Generate Noisy Sensor Measurements: y_k = theta_true * K_nominal * exp(N(0, sigma^2))
epsilons = np.random.normal(0, sigma, size=n_steps)
y_measurements = theta_true * K_nominal * np.exp(epsilons)

# 2. Prior Initialization: Beta(8, 1.5)
unnorm_prior = stats.beta.pdf(theta_grid, 8, 1.5)
Z_0 = np.trapezoid(unnorm_prior, theta_grid)
posterior_grid = unnorm_prior / Z_0

# Tracking containers
bayes_estimates = [np.trapezoid(theta_grid * posterior_grid, theta_grid)]
map_estimates = [theta_grid[np.argmax(posterior_grid)]]

# Store density profiles for visualization milestones
milestones = [0, 1, 2, 5, 10, 15]
density_profiles = {0: posterior_grid.copy()}

# 3. Sequential Non-Conjugate Grid Updates
for k in range(1, n_steps + 1):
    y_k = y_measurements[k - 1]

    # Compute Log-Normal Likelihood across grid
    mu_log = np.log(theta_grid * K_nominal)
    likelihood = (1.0 / (y_k * sigma * np.sqrt(2 * np.pi))) * np.exp(-((np.log(y_k) - mu_log) ** 2) / (2 * sigma ** 2))

    # Multiply prior and likelihood
    unnormalized_posterior = posterior_grid * likelihood

    # Normalize using Trapezoidal Rule
    Z_k = np.trapezoid(unnormalized_posterior, theta_grid)
    posterior_grid = unnormalized_posterior / Z_k

    # Calculate point estimators
    theta_bayes = np.trapezoid(theta_grid * posterior_grid, theta_grid)
    theta_map = theta_grid[np.argmax(posterior_grid)]

    bayes_estimates.append(theta_bayes)
    map_estimates.append(theta_map)

    if k in milestones:
        density_profiles[k] = posterior_grid.copy()

# -------------------------------------------------------------------------
# Plot 1: Evolution of Full Posterior Density Curves
# -------------------------------------------------------------------------
fig_densities = go.Figure()

colors = {0: 'gray', 1: 'purple', 2: 'orange', 5: 'green', 10: 'blue', 15: 'darkred'}

for m in milestones:
    fig_densities.add_trace(go.Scatter(
        x=theta_grid, y=density_profiles[m],
        mode='lines',
        name=f'Step k={m}',
        line=dict(color=colors[m], width=2.5 if m in [0, 15] else 1.8)
    ))

fig_densities.add_vline(x=theta_true, line_dash="dash", line_color="red", annotation_text="θ_true = 0.68")

fig_densities.update_layout(
    title="Structural Health Monitoring: Posterior Density Evolution Across Milestones",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)

fig_densities.show()

# -------------------------------------------------------------------------
# Plot 2: Convergence Timeline of Point Estimators
# -------------------------------------------------------------------------
steps = np.arange(0, n_steps + 1)

fig_timeline = go.Figure()

fig_timeline.add_trace(go.Scatter(
    x=steps, y=bayes_estimates,
    mode='lines+markers', name='Posterior Mean (θ_Bayes)',
    line=dict(color='blue', width=2.5), marker=dict(size=6)
))

fig_timeline.add_trace(go.Scatter(
    x=steps, y=map_estimates,
    mode='lines+markers', name='MAP Estimate (θ_MAP)',
    line=dict(color='darkorange', width=2, dash='dot'), marker=dict(size=6)
))

fig_timeline.add_trace(go.Scatter(
    x=[0, n_steps], y=[theta_true, theta_true],
    mode='lines', name='True Stiffness (θ_true = 0.68)',
    line=dict(color='red', width=2, dash='dash')
))

fig_timeline.update_layout(
    title="Sequential Estimation Convergence over Inspection Timeline",
    xaxis_title="Inspection Step (k)",
    yaxis_title="Stiffness Efficiency Factor (θ)",
    template="plotly_white",
    hovermode="x unified"
)

fig_timeline.show()

Analysis of Degradation Convergence & Safety Thresholds



*   Overcoming the Optimistic Prior: The system overcomes the initial optimistic prior ($\mathbb{E}[\Theta^{(0)}] \approx 0.84$) in 3 to 4 sensor readings. By step $k = 5$, both $\widehat{\theta}_{\mathrm{Bayes}}$ and $\widehat{\theta}_{\mathrm{MAP}}$ settle closely around $\theta_{\text{true}} = 0.68$.

*   Density Concentration (Certainty): The posterior density curve transitions from a broad prior profile ($k=0$) to a tall, narrow peak centered at $0.68$ by step $k=15$.

*  Implications for Structural Safety Thresholds:


    * Shrinking posterior variance directly narrows the Bayesian credible interval (e.g., $95\%$ High Posterior Density region).


   * In structural safety monitoring, if a critical maintenance threshold is set at $\theta_{\text{critical}} = 0.70$, the system can trigger an automated safety alert as soon as $P(\Theta < 0.70 \mid \mathbf{y}^{(k)}) > 0.95$. This framework provides probabilistic decision support before physical failure occurs.




